# Итоговая работа: прогноз спроса на велосипеды на ближайший час

ML Mentor · starter notebook · решения в репозитории отсутствуют.


## Задача

Постройте воспроизводимое базовое решение для прогноза спроса на прокат велосипедов на ближайший час и оформите его так, как если бы модель готовилась к первому запуску в продакшне. Каждая строка описывает один завершившийся час; при реальном применении признаки календаря и прогноз погоды формируются для следующего часа. Данные упорядочены по времени, поэтому случайное перемешивание здесь запрещено.

В таблице `cnt` — целевая переменная, а `casual` и `registered` — две части уже состоявшегося спроса. Их сумма равна `cnt`, поэтому использовать эти столбцы как признаки нельзя: в момент прогноза их ещё нет.

**Что должно получиться:** хронологическое разбиение на обучающую, проверочную и тестовую выборки (`train/validation/test`), три базовых решения, MAE и RMSLE, проверка утечки данных, срезы ошибок, краткая карточка модели, контракт инференса и три проверки сдвига данных.

Источник: UCI Bike Sharing Dataset, лицензия CC BY 4.0. Атрибуция и контрольные суммы находятся в `data/datasets.json`. CPU достаточно; API-ключи не нужны. В локальной копии репозитория используются уже сохранённые данные. Если открыть только notebook в Colab, подготовительная ячейка один раз скачает официальный ZIP по закреплённому адресу, проверит SHA-256 и дальше будет работать локально.

In [ ]:
import hashlib
import io
import time
import urllib.request
import zipfile
from pathlib import Path
from urllib.parse import urlparse

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_log_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy import sparse


UCI_SOURCE_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
UCI_SOURCE_SHA256 = "b70182d0d0508e9abbb79306ce5c0cec34869000f8220175ac83d11dbe845401"
HOUR_CSV_SHA256 = "e03de4ee4ef4dc376ac6e04bf829673c6269e8eba5c60fa121640fa2f829504f"


def sha256(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def resolve_bike_path() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        path = candidate / "data" / "uci-bike-sharing" / "hour.csv"
        if path.is_file():
            assert sha256(path.read_bytes()) == HOUR_CSV_SHA256
            return path

    cache_path = Path("/tmp/ml-mentor-labs-datasets-v1/uci-bike-sharing/hour.csv")
    if cache_path.is_file() and sha256(cache_path.read_bytes()) == HOUR_CSV_SHA256:
        return cache_path
    request = urllib.request.Request(UCI_SOURCE_URL, headers={"User-Agent": "ML-Mentor-Labs/1.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        assert urlparse(response.geturl()).hostname == "archive.ics.uci.edu"
        archive_payload = response.read()
    assert sha256(archive_payload) == UCI_SOURCE_SHA256, "UCI source checksum changed"
    with zipfile.ZipFile(io.BytesIO(archive_payload)) as archive:
        hour_payload = archive.read("hour.csv")
    assert sha256(hour_payload) == HOUR_CSV_SHA256
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_bytes(hour_payload)
    return cache_path


BIKE_PATH = resolve_bike_path()
TARGET = "cnt"
LEAKAGE_COLUMNS = {"casual", "registered"}
NON_FEATURE_COLUMNS = {"instant", "dteday", TARGET, *LEAKAGE_COLUMNS}
CATEGORICAL_FEATURES = ["season", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]
NUMERIC_FEATURES = ["yr", "temp", "atemp", "hum", "windspeed"]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

df = pd.read_csv(BIKE_PATH, parse_dates=["dteday"]).sort_values(["dteday", "hr"]).reset_index(drop=True)
df["timestamp"] = df["dteday"] + pd.to_timedelta(df["hr"], unit="h")
assert df["timestamp"].is_monotonic_increasing
assert (df["casual"] + df["registered"] == df[TARGET]).all()
assert not (set(FEATURES) & NON_FEATURE_COLUMNS)
assert df[TARGET].ge(0).all()


## 1. Хронологическая проверка

Отделите последние 15% наблюдений в тестовую выборку, предшествующие 15% — в проверочную, остальное — в обучающую. Границы должны зависеть только от позиции во временном ряду. Тестовую выборку не используйте при выборе признаков, модели и гиперпараметров.

In [ ]:
def chronological_split(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # TODO: верните непересекающиеся train, validation и test в хронологическом порядке.
    raise NotImplementedError


train_df, validation_df, test_df = chronological_split(df)


## 2. Три базовых решения в одном бюджете

Сравните на одном разбиении и одних признаках три решения:

1. сезонный наивный прогноз — значение спроса ровно 168 часов назад в последовательной проверке;
2. Ridge с отдельной обработкой числовых и категориальных признаков;
3. `HistGradientBoostingRegressor(random_state=42)`.

У обучаемых моделей одинаковый бюджет: одно обучение без поиска по широкой сетке. `HistGradientBoostingRegressor` не принимает разреженную матрицу, поэтому его `OneHotEncoder` должен использовать `sparse_output=False` (либо нужна другая корректная плотная предобработка). Любые преобразования обучаются только на обучающей выборке.

Для сезонного прогноза разрешено использовать целевую переменную из проверочной или тестовой выборки только тогда, когда соответствующий час уже наступил: для прогноза в момент `t` источник обязан быть ровно `t − 168h` и строго раньше `t`. Это имитирует почасовую последовательную проверку (rolling backtest). Если брать историю только из обучающей выборки, для поздних недель базовый прогноз ошибочно исчезнет.

Считайте MAE в исходных единицах спроса и RMSLE после обрезки отрицательных прогнозов до нуля. RMSLE сильнее наказывает относительную ошибку в часы с небольшим спросом.

In [ ]:
MODEL_NAMES = ("ridge", "hist_gradient_boosting")


def seasonal_naive(observations: pd.DataFrame, prediction_rows: pd.DataFrame) -> tuple[np.ndarray, pd.DataFrame]:
    # TODO: сопоставьте каждому prediction_timestamp источник ровно t - 168h.
    # Верните predictions и audit columns=[prediction_timestamp, source_timestamp, source_value].
    # observations может содержать весь backtest, но lookup имеет право читать только source_timestamp < prediction_timestamp.
    raise NotImplementedError


def build_model(name: str) -> Pipeline:
    # TODO: соберите Pipeline с шагами preprocess и model.
    # Ridge может получать sparse one-hot, HistGradientBoosting обязан получать dense matrix.
    raise NotImplementedError


def regression_report(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    # TODO: верните mae и rmsle; для RMSLE обрежьте прогноз снизу нулём.
    raise NotImplementedError


models = {name: build_model(name) for name in MODEL_NAMES}
# TODO: получите predictions всех трёх baseline на validation и одну строку метрик на модель.
validation_predictions = {}
validation_results = pd.DataFrame(columns=["model", "mae", "rmsle", "fit_seconds"])
seasonal_validation_audit = None
# TODO: выберите learned-модель по validation; naive остаётся обязательной точкой сравнения.
SELECTED_MODEL_NAME = None
PRODUCTION_DECISION = None


## 3. Финальная проверка

После фиксации решения переобучите выбранный конвейер на обучающей и проверочной выборках и ровно один раз посчитайте метрики на тестовой. Не подбирайте параметры по тестовому результату. Для выбранной модели посчитайте срезы MAE на проверочной и тестовой выборках по часу суток, признаку рабочего дня и погодной категории. Небольшой выигрыш общей метрики может скрывать систематический провал на важном сегменте.

Сохраните обученный конвейер через `joblib` и явно опишите входную схему артефакта.

In [ ]:
# TODO: после фиксации решения получите final_model, test_predictions и test_metrics.
final_model = None
test_predictions = None
test_metrics = None
ARTIFACT_PATH = Path("/tmp/ml-mentor-bike-demand.joblib")
# TODO: сериализуйте final_model в ARTIFACT_PATH.
INPUT_SCHEMA = [
    # TODO: по одному dict(name, dtype, nullable) на каждый признак из FEATURES.
]


def mae_slices(frame: pd.DataFrame, y_true: pd.Series, y_pred: np.ndarray, *, split: str) -> pd.DataFrame:
    # TODO: верните columns=[split, dimension, value, n, mae].
    # Имена dimension: hr -> hour, workingday -> workingday, weathersit -> weather.
    raise NotImplementedError


validation_slices = None
test_slices = None


## 4. Контракт и наблюдаемость

Заполните структуры ниже короткими проверяемыми формулировками. Карточка модели должна объяснять назначение и границы базового решения. Контракт инференса перечисляет входы, выход и поведение при ошибке. Для каждой проверки сдвига данных укажите сигнал, окно наблюдения и порог реакции.

In [ ]:
# TODO: замените None конкретными значениями и формулировками.
MODEL_CARD = {
    "purpose": None,
    "target_horizon": None,
    "validation_scheme": None,
    "primary_metric": None,
    "known_limitations": None,
}
INFERENCE_CONTRACT = {
    "required_features": None,
    "prediction_type": None,
    "invalid_input_policy": None,
}
DRIFT_CHECKS = [
    # TODO: минимум три dict с keys signal, window, threshold, action.
]


In [ ]:
# Self-check: запускайте после выполнения всех TODO.
assert len(train_df) + len(validation_df) + len(test_df) == len(df)
assert train_df.index.is_unique and validation_df.index.is_unique and test_df.index.is_unique
assert train_df["timestamp"].max() < validation_df["timestamp"].min() < test_df["timestamp"].min()
assert not (set(FEATURES) & LEAKAGE_COLUMNS)
assert set(models) == set(MODEL_NAMES)
assert set(validation_predictions) == {"seasonal_naive", *MODEL_NAMES}
assert all(len(prediction) == len(validation_df) for prediction in validation_predictions.values())
assert seasonal_validation_audit is not None
assert set(seasonal_validation_audit.columns) == {"prediction_timestamp", "source_timestamp", "source_value"}
assert len(seasonal_validation_audit) == len(validation_df)
assert seasonal_validation_audit["prediction_timestamp"].reset_index(drop=True).equals(
    validation_df["timestamp"].reset_index(drop=True)
)
assert (seasonal_validation_audit["source_timestamp"] < seasonal_validation_audit["prediction_timestamp"]).all()
assert (
    seasonal_validation_audit["prediction_timestamp"] - seasonal_validation_audit["source_timestamp"]
).eq(pd.Timedelta(hours=168)).all()
assert seasonal_validation_audit["source_value"].notna().all()
assert np.allclose(validation_predictions["seasonal_naive"], seasonal_validation_audit["source_value"])
assert all({"preprocess", "model"} <= set(model.named_steps) for model in models.values())
ridge_matrix = models["ridge"].named_steps["preprocess"].transform(validation_df[FEATURES].head(4))
hist_matrix = models["hist_gradient_boosting"].named_steps["preprocess"].transform(validation_df[FEATURES].head(4))
assert sparse.issparse(ridge_matrix) or isinstance(ridge_matrix, np.ndarray)
assert isinstance(hist_matrix, np.ndarray) and not sparse.issparse(hist_matrix)
assert set(validation_results["model"]) == {"seasonal_naive", *MODEL_NAMES}
assert {"model", "mae", "rmsle", "fit_seconds"} <= set(validation_results.columns)
assert validation_results[["mae", "rmsle", "fit_seconds"]].ge(0).all().all()
assert SELECTED_MODEL_NAME in MODEL_NAMES
assert isinstance(PRODUCTION_DECISION, str) and PRODUCTION_DECISION.strip()
assert final_model is not None and test_predictions is not None
assert len(test_predictions) == len(test_df)
assert set(test_metrics) == {"mae", "rmsle"}
assert all(np.isfinite(list(test_metrics.values())))
assert ARTIFACT_PATH.is_file()
restored_model = joblib.load(ARTIFACT_PATH)
assert len(restored_model.predict(test_df[FEATURES].head(2))) == 2
assert {item["name"] for item in INPUT_SCHEMA} == set(FEATURES)
assert all({"name", "dtype", "nullable"} == set(item) for item in INPUT_SCHEMA)
assert all(item["dtype"] == str(df[item["name"]].dtype) for item in INPUT_SCHEMA)
assert all(isinstance(item["nullable"], bool) for item in INPUT_SCHEMA)
expected_slice_columns = {"split", "dimension", "value", "n", "mae"}
assert validation_slices is not None and set(validation_slices.columns) == expected_slice_columns
assert test_slices is not None and set(test_slices.columns) == expected_slice_columns
assert set(validation_slices["dimension"]) == {"hour", "workingday", "weather"}
assert set(test_slices["dimension"]) == {"hour", "workingday", "weather"}
assert set(validation_slices["split"]) == {"validation"} and set(test_slices["split"]) == {"test"}
assert validation_slices[["n", "mae"]].ge(0).all().all()
assert test_slices[["n", "mae"]].ge(0).all().all()
assert validation_slices.groupby("dimension")["n"].sum().eq(len(validation_df)).all()
assert test_slices.groupby("dimension")["n"].sum().eq(len(test_df)).all()
assert all(MODEL_CARD.values()) and all(INFERENCE_CONTRACT.values())
assert len(DRIFT_CHECKS) >= 3
assert all({"signal", "window", "threshold", "action"} <= set(item) for item in DRIFT_CHECKS)
print({"validation": validation_results, "test": test_metrics})
